# Part 4
In this part we will make our model more in-line with our submission-format, mainly by transforming to cumulative weights.


In [1]:
#Imports
import pandas as pd
import xgboost as xgb
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

#Our dataframe
final_df = pd.read_csv("cleaned_data/cleaned_df_final_test.csv") #TEMP NAME, remember to change.
#Variables that we need

In [5]:
#Define what proportion should be reserved for verifying. Evaluation will be done with the latter part.
SPLIT_RATIO = 0.975

#Find what day this corresponds to
split_day_numerical = final_df["days_normalized"].quantile(SPLIT_RATIO)
print("Splitting at ", SPLIT_RATIO, ", Corresponds to day", split_day_numerical)

#Split data according to our index
train_df = final_df[final_df["days_normalized"] <= split_day_numerical]
test_df = final_df[final_df["days_normalized"] > split_day_numerical]
#Verify the split

print(f"Train range: {train_df['days_normalized'].min()} to {train_df['days_normalized'].max()}")
print(f"Test range: {test_df['days_normalized'].min()} to {test_df['days_normalized'].max()}")

first_test_date = test_df["date_str"].iloc[0]
last_test_date = test_df["date_str"].iloc[-1]
print("The data used for testings starts at ", first_test_date, ", and ends at ", last_test_date)


Splitting at  0.975 , Corresponds to day 7305.0
Train range: 0 to 7305
Test range: 7306 to 7492
The data used for testings starts at  2024-06-16 , and ends at  2024-12-19


Create a cumulative sum for test data to verify the training data.


In [21]:
valid_ids = final_df["rm_id"].unique()

cumsums = []
for id in valid_ids:
    temp_df = test_df[test_df["rm_id"] == id]
    cumsum = temp_df["daily_weight"].cumsum()
    cumsums.extend(cumsum.to_list())


test_df["cumulative_weight"] = cumsums



C:\Users\aksel\AppData\Local\Temp\ipykernel_14184\3157555919.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_df["cumulative_weight"] = cumsums


Convert predictions to cumsums as well:

In [22]:
test_predictions = pd.read_csv("test_predictions/test_predictions_new.csv")
#test_predictions = pd.read_csv("test_predictions_simple.csv")

In [23]:

cumsums =[]
for id in valid_ids:
    temp_df = test_predictions[test_predictions["rm_id"] == id]
    cumsum = temp_df["predicted_weight"].cumsum()
    cumsums.extend(cumsum.to_list())


test_predictions["cumulative_weight"] = cumsums

Evaluate performance:

In [ ]:
def quantile_loss(predictions, actual):
    quantile_losses = []
    for ids in valid_ids:
        id_predictions = predictions[predictions["rm_id"] == ids]
        #Valid: "cumulative_weight", "flat_scaled_predictions", "linear_scaled_predictions", "curved_scaled_predictions"
        id_predictions = id_predictions["cumulative_weight"].sum()
        actual_predictions = actual[actual["rm_id"]  == ids]
        actual_predictions = actual_predictions["cumulative_weight"].sum()

        id_loss = max( 0.2 * (actual_predictions - id_predictions), 0.8 * (id_predictions - actual_predictions))
        quantile_losses.append(id_loss)
        
    avg_quantile_loss = (1/len(valid_ids) * sum(quantile_losses))
    return(avg_quantile_loss)

In [24]:
predictions = test_predictions["cumulative_weight"]
actual = test_df["cumulative_weight"]

mse = mean_squared_error(predictions, actual)
rmse = np.sqrt(mse)
mae = mean_absolute_error(predictions, actual)
r2 = r2_score(predictions, actual)
q08 = quantile_loss(test_predictions, test_df)

print(f'Mean squared Error: {mse:.4f}')
print(f'Root Mean Squared: {rmse:.4f}')
print(f'Mean absolute error: {mae:.4f}')
print(f'r2-score: {r2:.4f}')

print(f'Quantile loss: {q08:.4f}')

KeyError: 'flat_scaled_predictions'

## 4.2 Custom transformations
Apply a custom transformation to see if we can introduce a bias since we did not manage to use quantileerror for predictions.

### Flat scaling

In [20]:
SCALING_FACTOR = 0.5
test_predictions["flat_scaled_predictions"] = test_predictions["cumulative_weight"] * SCALING_FACTOR
predictions = test_predictions["flat_scaled_predictions"]
actual = test_df["cumulative_weight"]
r2 = r2_score(predictions, actual)
q08 = quantile_loss(test_predictions, test_df)
print("Flat Scaling: ", SCALING_FACTOR)
print(f'r2-score: {r2:.4f}')

print(f'Quantile loss: {q08:.4f}')


Flat Scaling:  0.5
r2-score: -0.6890
Quantile loss: 2474637.6792


### Progressive scaling: Linear

In [98]:
x = 0
for i in range(0, len(test_predictions)):
    if test_predictions["rm_id"].iloc[i] == 365:
        x += 1

lower_bound = -0.3
upper_bound = 0.9
step_size = ((upper_bound-lower_bound)/x)

factors = np.arange(lower_bound,upper_bound,step_size)
linear_scaled_list = []
for id in valid_ids:
    temp_id_group = test_predictions[test_predictions["rm_id"] == id]
    aea = (temp_id_group["cumulative_weight"] * factors).to_list()
    linear_scaled_list.extend(aea)
test_predictions["linear_scaled_predictions"] = linear_scaled_list

predictions = test_predictions["linear_scaled_predictions"]
actual = test_df["cumulative_weight"]
r2 = r2_score(predictions, actual)
q08 = quantile_loss(test_predictions, test_df)
print("Linear scaling: ", lower_bound, " to ", upper_bound)
print(f'r2-score: {r2:.4f}')

print(f'Quantile loss: {q08:.4f}')




Linear scaling:  -0.3  to  0.9
r2-score: -0.1226
Quantile loss: 2439090.5310


### curved scaling:
this will scale down earlier predictions more than later predictions.

In [136]:
def nonlinear_space(start, stop, num, curvature=1.0):

    # Generate evenly spaced values between 0 and 1
    t = np.arange(start, stop, num)
    # Apply curvature (a power function)
    t = t ** curvature

    return t


x = 0
for i in range(0, len(test_predictions)):
    if test_predictions["rm_id"].iloc[i] == 365:
        x += 1
lower_bound = 0.2
upper_bound = 1
step_size = ((upper_bound-lower_bound)/x)
curve = 2

factors = nonlinear_space(lower_bound, upper_bound, step_size, curve)

curved_scaled_list = []
for id in valid_ids:
    temp_id_group = test_predictions[test_predictions["rm_id"] == id]
    aea = (temp_id_group["cumulative_weight"] * factors).to_list()
    curved_scaled_list.extend(aea)
test_predictions["curved_scaled_predictions"] = curved_scaled_list

predictions = test_predictions["curved_scaled_predictions"]
actual = test_df["cumulative_weight"]
r2 = r2_score(predictions, actual)
q08 = quantile_loss(test_predictions, test_df)
print("Curved scaling: ", lower_bound, " to ", upper_bound, " with factor: ", curve)
print(f'r2-score: {r2:.4f}')

print(f'Quantile loss: {q08:.4f}')

Curved scaling:  0.2  to  1  with factor:  2
r2-score: 0.1344
Quantile loss: 2478045.7751
